In [1]:
import pandas as pd

roll_number = "1024170256" 

last_two_digits = [int(d) for d in str(roll_number)[-2:]]
print(f"Using last two digits: {last_two_digits}")

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

# Pool of realistic custom entries to draw from dynamically
custom_pool = {
    "billing": [
        {"question": "how do i get a refund", "answer": "Email support for refund requests.", "keywords": "refund return money"},
        {"question": "where is my invoice", "answer": "Invoices are available in your dashboard.", "keywords": "invoice bill statement"}
    ],
    "account": [
        {"question": "how do i update my mobile number", "answer": "Go to Profile Settings > Mobile.", "keywords": "mobile number phone update"},
        {"question": "how to delete account", "answer": "Account deletion is under privacy settings.", "keywords": "delete remove cancel"}
    ],
    "general": [
        {"question": "where is the head office", "answer": "We are located in New Delhi.", "keywords": "office location address"},
        {"question": "do you have a mobile app", "answer": "Yes, available on Android and iOS.", "keywords": "app mobile download"}
    ]
}

categories_map = ["billing", "account", "general"]
custom_entries = []
cat_usage_tracker = {"billing": 0, "account": 0, "general": 0}

# Construct the 2 personalized entries
for d in last_two_digits:
    assigned_category = categories_map[d % 3]
    
    # Pick a realistic entry from the pool (cycles through if category repeats)
    pool_index = cat_usage_tracker[assigned_category] % len(custom_pool[assigned_category])
    new_entry = custom_pool[assigned_category][pool_index].copy()
    new_entry["category"] = assigned_category
    
    custom_entries.append(new_entry)
    cat_usage_tracker[assigned_category] += 1

# Build final DataFrame
df = pd.DataFrame(fixed_entries + custom_entries)

print("\n--- Output Q1: Final 6-row DataFrame ---")
display(df) # Use print(df) if not running inside Jupyter

Using last two digits: [5, 6]

--- Output Q1: Final 6-row DataFrame ---


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where is the head office,We are located in New Delhi.,office location address,general
5,how do i get a refund,Email support for refund requests.,refund return money,billing


In [3]:
def score_hypothesis(query, dataframe):
    """
    Scores each entry based on how many words from the query match the keywords.
    Returns all matching entries ranked by confidence (score).
    """
    query_words = set(query.lower().split())
    scores = []
    
    for idx, row in dataframe.iterrows():
        kw_words = set(row['keywords'].lower().split())
        # Calculate intersection to find matching words
        score = len(query_words.intersection(kw_words))
        scores.append(score)
        
    df_scored = dataframe.copy()
    df_scored['score'] = scores
    
    # Filter out zero scores and sort by highest confidence
    matches = df_scored[df_scored['score'] > 0].sort_values(by='score', ascending=False)
    return matches

# Quick test of Q2
print("--- Output Q2: Testing scoring function with query 'fee upi' ---")
display(score_hypothesis("how do i pay the fee via upi", df))

--- Output Q2: Testing scoring function with query 'fee upi' ---


,question,answer,keywords,category,score
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,3
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,1


In [4]:
def same_category(category_name, dataframe):
    """Returns a list of all questions belonging to the given category."""
    # Filter by category and extract the 'question' column to a list
    filtered_questions = dataframe[dataframe['category'] == category_name]['question'].tolist()
    return filtered_questions

# Call it using the category of the FIRST personalized entry from Q1
chosen_category = custom_entries[0]['category']

print(f"--- Output Q3: All questions in category '{chosen_category}' ---")
questions_list = same_category(chosen_category, df)
for q in questions_list:
    print("-", q)

--- Output Q3: All questions in category 'general' ---
- what are your working hours
- where is the head office


In [5]:
# Pick the first entry in the knowledge base (Index 0)
target_index = 0

print(f"Target entry question: '{df.loc[target_index, 'question']}'")
print(f"Current keywords: '{df.loc[target_index, 'keywords']}'")

# Ask user for input
new_keyword = input("Enter a new keyword to add to this entry: ")

# Append the new keyword
df.loc[target_index, 'keywords'] = df.loc[target_index, 'keywords'] + " " + new_keyword.strip()

print(f"\nUpdated keywords: '{df.loc[target_index, 'keywords']}'")

# Save to CSV using the roll number formatting requested
csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print(f"\n--- Output Q4: Saved updated DataFrame to {csv_filename} ---")

Target entry question: 'what is the annual fee'
Current keywords: 'fee cost price charge'


Enter a new keyword to add to this entry:  price charge ksndjojiojsx bkjskjbskxs



Updated keywords: 'fee cost price charge price charge ksndjojiojsx bkjskjbskxs'

--- Output Q4: Saved updated DataFrame to 1024170256_faq_data.csv ---


In [6]:
print("--- Output Q5: FAQ entries per category ---")

# Using groupby to count occurrences of each category
category_counts = df.groupby('category').size()
print(category_counts)

--- Output Q5: FAQ entries per category ---
category
account    1
billing    3
general    2
dtype: int64


In [7]:
def score_hypothesis_with_ties(query, dataframe):
    """
    Modified scoring function that handles ties. If multiple entries 
    share the highest score, it prints all of them instead of picking one.
    """
    # Reuse the logic from Q2 to get sorted matches
    matches = score_hypothesis(query, dataframe)
    
    if matches.empty:
        print(f"Query: '{query}' -> No matches found.")
        return None
        
    # Find the maximum score
    max_score = matches['score'].max()
    
    # Filter for ONLY the entries that have this maximum score
    best_matches = matches[matches['score'] == max_score]
    
    if len(best_matches) > 1:
        print(f"Query: '{query}' -> TIE DETECTED! {len(best_matches)} entries tied with the highest score of {max_score}:")
        display(best_matches[['question', 'keywords', 'category', 'score']])
    else:
        print(f"Query: '{query}' -> Single best match found with score of {max_score}:")
        display(best_matches[['question', 'keywords', 'category', 'score']])
        
    return best_matches

print("--- Output Q6: Demonstrating Ties ---")

# 1. Query that produces a tie (Matching both "fee" entries)
score_hypothesis_with_ties("fee", df)

print("\n" + "="*50 + "\n")

# 2. Query that does NOT produce a tie (Matches only the password entry)
score_hypothesis_with_ties("password", df)

--- Output Q6: Demonstrating Ties ---
Query: 'fee' -> TIE DETECTED! 2 entries tied with the highest score of 1:


,question,keywords,category,score
0,what is the annual fee,fee cost price charge price charge ksndjojiojs...,billing,1
3,how can i pay the fee,pay payment upi fee,billing,1




Query: 'password' -> Single best match found with score of 1:


,question,keywords,category,score
1,how to reset password,password reset login,account,1


,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,1
